# Trabalho 2 - CG

### Imports

In [230]:
!pip install pyopengl
!pip install glfw
!pip install pyglm
!pip install numpy
!pip install pillow

In [231]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image
import json

from shader_s import Shader

### Inicializando janela

In [232]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [233]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [234]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

### Controlador de cena

In [235]:
# Carrega os dados da cena a partir de um arquivo .json
def load_scene():
    scene = []
    with open("scene.json", "r", encoding="utf-8") as f:
        scene = json.load(f)
    
    return scene

# Salva os dados da cena em um arquivo .json
def save_scene(scene):
    with open("scene.json", "w", encoding="utf-8") as f:
        json.dump(scene, f, ensure_ascii=False, indent=4)

# Adiciona um objeto a cena
def add_to_scene(scene, obj_id, name, obj_file, texture_file):
    
    newObj = {
        "name": name,
        "obj_file": obj_file,
        "texture_file": texture_file,
        "angulo_obj": 0.0,
        "translacao": [
            2.0,
            0.0,
            -20.0
        ],
        "escala": [
            1.5,
            1.5,
            1.5
        ],
        "rotacao": [
            0.0,
            0.0,
            1.0
        ],
        "id_textura": obj_id,
        "obj_id": obj_id,
        "visible": True,
        "polygon": False,
        "scene_id": len(scene),
        # "draw_mode": objects_vex[obj_id][3],
        # "group_id": -1
    }

    scene.append(newObj)

    return scene

### Criar objeto

In [ ]:
scene = load_scene() # Carrega a cena
def criar_objeto(scene, obj_file, texture_file, name):

    add_to_scene(scene, len(scene), name, obj_file, texture_file)

    save_scene(scene)


general_name = "spiderman"
texture_type = "png"

obj_file = f'objetos/{general_name}/{general_name}.obj'
texture_file = f'objetos/{general_name}/{general_name}.{texture_type}'
criar_objeto(scene, obj_file, texture_file, general_name)

### Carregar modelos

In [237]:
verticeInicial_quantosVertices_list = []

for object in scene:
    verticeInicial, quantosVertices = load_obj_and_texture(object['obj_file'], [object['texture_file']])
    verticeInicial_quantosVertices_list.append([verticeInicial, quantosVertices])

Processando modelo objetos/grama/grama.obj. Vertice inicial: 0
Processando modelo objetos/grama/grama.obj. Vertice final: 149760
0
Processando modelo objetos/spiderman/spiderman.obj. Vertice inicial: 149760
Processando modelo objetos/spiderman/spiderman.obj. Vertice final: 599742
1
Processando modelo objetos/carro/carro.obj. Vertice inicial: 599742
Processando modelo objetos/carro/carro.obj. Vertice final: 18835836
2


### Vamos carregar cada modelo e definir funções para desenhá-los

In [238]:
# # carrega caixa (modelo e texturas)
# verticeInicial_caixa, quantosVertices_caixa = load_obj_and_texture('objetos/caixa/caixa.obj', ['objetos/caixa/caixa.jpg'])
# verticeInicial_spider_man, quantosVertices_spider_man = load_obj_and_texture('objetos/spiderman/spiderman.obj', ['objetos/spiderman/spiderman.png'])
# verticeInicial_grama, quantosVertices_grama = load_obj_and_texture('objetos/grama/grama.obj', ['objetos/grama/grama.jpg'])



def desenha_obj(angle, rot_coords, transl_coords, escal_coords, textureId, verticeInicial_obj, quantosVertices_obj):
    
    r_x, r_y, r_z = rot_coords
    t_x, t_y, t_z = transl_coords
    s_x, s_y, s_z = escal_coords

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_obj, quantosVertices_obj) ## renderizando

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [239]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [240]:
if(len(scene) != 0):
    vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
    vertices['position'] = vertices_list


    # Upload data
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices = glGetAttribLocation(program, "position")
    glEnableVertexAttribArray(loc_vertices)
    glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [241]:
if(len(scene) != 0):
    textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
    textures['position'] = textures_coord_list


    # Upload data
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
    glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
    stride = textures.strides[0]
    offset = ctypes.c_void_p(0)
    loc_texture_coord = glGetAttribLocation(program, "texture_coord")

    glEnableVertexAttribArray(loc_texture_coord)
    glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Eventos para modificar a posição da câmera.

* Usei as teclas H, B, N, M para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [242]:
eixo = 1 # Eixo selecionado para rotação/escala
selected_obj = 0 # Objeto selecionado para a manipulação
uniform_scale = True # Toggle para escala uniforme
PolygonMode = False # Toggle para modo polígono
groupMode = True # Toggle para modo de grupo (usado para debugging, não pode ser desativado pelo teclado)
restrictMode = False # Toggle para modo restrito (usado para debugging, não pode ser desativado pelo teclado)

# camera
cameraPos   = glm.vec3(0.0, 0.0, 3.0)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2


def key_event(window,key,scancode,action,mods):
    global eixo, selected_obj, scene, uniform_scale, PolygonMode, groups, restrictMode
    global cameraPos, cameraFront, cameraUp

    # Manipulações de grupo exigidas pelo trabalho
    if restrictMode:
        print("restrict mode")
        # Toggle Polygon Mode
        # if key == glfw.KEY_P and action == glfw.PRESS: 
        #     PolygonMode = not PolygonMode
        
        # # Rotação (Peixe)
        # if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[2]['rotacao'][2] += 0.05
        # if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[2]['rotacao'][2] -= 0.05    

        # # Escala (Nuvem)
        # if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[1]['escala'][0] += 0.01
        #     groups[1]['escala'][1] += 0.01
        #     groups[1]['escala'][2] += 0.01
        # if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        #     if groups[1]['escala'][0] >= 0:
        #         groups[1]['escala'][0] -= 0.01
        #         groups[1]['escala'][1] -= 0.01
        #         groups[1]['escala'][2] -= 0.01
        
        # # Translação (Sol)
        # ## X
        # if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[0]['deslocamento'][0] -= 0.01
        # if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[0]['deslocamento'][0] += 0.01
        # ## Y
        # if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[0]['deslocamento'][1] -= 0.01
        # if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
        #     groups[0]['deslocamento'][1] += 0.01
        
    # Manipulações individuais utilizadas na montagem da cena
    else:
        # Muda Objeto
        if key == glfw.KEY_Y and action == glfw.PRESS:
            selected_obj = (selected_obj+1) % len(scene)

        # Toggle Polygon Mode
        if key == glfw.KEY_P and action == glfw.PRESS: 
            PolygonMode = not PolygonMode

        # Toggle Visibility
        if key == glfw.KEY_V and action == glfw.PRESS: 
            scene[selected_obj]['visible'] = not scene[selected_obj]['visible']

        # Toggle uniform_scale
        if key == glfw.KEY_C and action == glfw.PRESS:
            uniform_scale = not uniform_scale

        # Troca de Eixo Rotação
        if key == glfw.KEY_Q and action == glfw.PRESS:
            eixo = (eixo+1)%3

        # Rotação
        if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] += 0.5
        if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] -= 0.5

        # Escala
        if uniform_scale:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] += 0.01
                scene[selected_obj]['escala'][1] += 0.01
                scene[selected_obj]['escala'][2] += 0.01
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] -= 0.01
                scene[selected_obj]['escala'][1] -= 0.01
                scene[selected_obj]['escala'][2] -= 0.01
        else:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] += 0.01
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] -= 0.01

        # Translação
        ## X
        if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][0] -= 0.5
        if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][0] += 0.5
        ## Y
        if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][1] -= 0.5
        if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][1] += 0.5
        ## Z
        if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][2] -= 0.5
        if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][2] += 0.5



        

        if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
            glfw.set_window_should_close(window, True)
        
        cameraSpeed = 200 * deltaTime
        if key == glfw.KEY_H and (action == glfw.PRESS or action == glfw.REPEAT):
            cameraPos += cameraSpeed * cameraFront
        
        if key == glfw.KEY_N and (action == glfw.PRESS or action == glfw.REPEAT):
            cameraPos -= cameraSpeed * cameraFront
        
        if key == glfw.KEY_B and (action == glfw.PRESS or action == glfw.REPEAT):
            cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
            
        if key == glfw.KEY_M and (action == glfw.PRESS or action == glfw.REPEAT):
            cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [243]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    # r_x, r_y, r_z são ângulos de Euler (em graus) em torno de X, Y, Z.
    # O parâmetro `angle` é mantido por compatibilidade com a assinatura,
    # mas não é mais usado (cada eixo tem seu próprio ângulo).

    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade

    # Ordem aplicada aos vértices: Scale -> Rotate -> Translate
    # (em GLM/coluna-major, basta multiplicar nesta ordem: T * R * S)

    # aplicando translacao
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))

    # aplicando rotacao em cada eixo (ângulos de Euler X, Y, Z)
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_x), glm.vec3(1.0, 0.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_y), glm.vec3(0.0, 1.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_z), glm.vec3(0.0, 0.0, 1.0))

    # aplicando escala
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))

    matrix_transform = np.array(matrix_transform)

    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Nesse momento, nós exibimos a janela!


In [244]:
glfw.show_window(window)

In [245]:
# id_caixa = 0
# id_aranha = 3
# id_grama = 4

### Loop principal da janela.

In [246]:
global scene
glEnable(GL_DEPTH_TEST) ### importante para 3D

draw_mode = [GL_TRIANGLE_STRIP, GL_TRIANGLE_FAN] # Modos de desenho utilizados

scene = load_scene() # Carrega a cena




# Loop principal, continua enquanto a janela estiver aberta
while not glfw.window_should_close(window):
   
    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)

    # Para cada objeto na cena, realiza a lógica de renderização
    for i, object in enumerate(scene):

        # Controla o modo polígono
        if PolygonMode:
            glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
        else:
            glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

        # Ajusta nome da janela de acordo com o modo restrito
        if restrictMode:
            glfw.set_window_title(window, "Praia")
        else:
            glfw.set_window_title(window, f"name: {scene[selected_obj]['name']}, scene_id: {scene[selected_obj]['scene_id']}, uniform_scale: {uniform_scale}, eixo: {eixo}")

        # Pula a renderização do objeto se ele estiver marcado como 'não-visível'
        if not object['visible']:
            continue

        # Pegando valores do objeto
        vertice_inicial = verticeInicial_quantosVertices_list[i][0]
        quantos_vertices = verticeInicial_quantosVertices_list[i][1]
        angulo_obj = object["angulo_obj"]
        ax, ay, az = object['rotacao']
        tx, ty, tz = object['translacao']
        sx, sy, sz = object['escala']
        id_textura = object['id_textura']

        # angulo, rotacao(x, y, z), translacao(x, y, z), escala(x, y, z), ID_textura
        desenha_obj(angulo_obj, [ax, ay, az], [tx, ty, tz], [sx, sy, sz], id_textura, vertice_inicial, quantos_vertices)






    #     # desenha_obj(0.0, [0, 0, 1], [2, 0, -20], [1.5, 1.5, 1.5], id_aranha, verticeInicial_spider_man, quantosVertices_spider_man)
    #     # desenha_obj(180.0, [0, 1, 1], [2, -20, -20], [0.5, 0.5, 0.5], id_grama, verticeInicial_grama, quantosVertices_grama)
        
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)


glfw.terminate()

save_scene(scene) # Salva a cena (apenas objetos individuais, não grupos)